In [1]:
%load_ext watermark
%watermark -a "Manuel Elias Orellana Lavayen" -d -v -iv

Author: Manuel Elias Orellana Lavayen

Date: 2026-10-05

Python implementation: CPython
Python version       : 3.12.10
IPython version      : 9.17.1

debugpy: 1.8.22



## Importando Librerias

In [2]:
import pandas as pd
from pathlib import Path
ruta_base = Path.cwd().parent.parent
print(ruta_base)

C:\Users\ManuelOrellana\Desktop\Proyectos Aplicaciones\Aplicacion Prediccion Productos\Creacion de Modelos\Analisis y entrenamiento de modelos


### Cargando Dataset

In [4]:
df_holidays_events = pd.read_csv(ruta_base / 'Datasets/Originales/holidays_events.csv')
df_holidays_events.head()

,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False


### Creando Datasets en base a tipos de eventos

In [5]:
tupla_feriados = ("Holiday", "Additional", "Transfer", "Bridge")

df_holidays_events["es_feriado"] = df_holidays_events["type"].isin(tupla_feriados) & (df_holidays_events["transferred"] == False)
df_holidays_events["es_evento"] = df_holidays_events["type"] == "Event"

df_holidays_events = df_holidays_events.drop(columns= ["type", "transferred", "description"])


# --- NACIONAL ---
df_holidays_nacional = df_holidays_events[df_holidays_events["locale"] == "National"]
df_holidays_nacional = df_holidays_nacional.drop(columns=["locale", "locale_name"])
df_holidays_nacional = df_holidays_nacional.rename(columns= {"es_feriado": "es_feriado_nacional",
                                                             "es_evento": "es_evento"})

# AGRUPAR POR FECHA PARA EVITAR DUPLICADOS (Si no se agrupan, se pueden crear dos filas con la misma fecha, una con True en evento y la otra con True en feriado)
df_holidays_nacional = (
    df_holidays_nacional.groupby("date")
    .agg({"es_feriado_nacional": "max", "es_evento": "max"}) # Agarramos el mayor (True= 1, False= 0)
    .reset_index() # Deshacemos el groupby
)

print("Nacional")
print(df_holidays_nacional["es_feriado_nacional"].unique())
print(df_holidays_nacional["es_evento"].unique())

df_holidays_nacional.to_csv(ruta_base / "Datasets/Modificados/holidays_nacional.csv", index=False)


# --- REGIONAL ---
df_holidays_regional = df_holidays_events[df_holidays_events["locale"] == "Regional"]
df_holidays_regional = df_holidays_regional.drop(columns=["locale"])
df_holidays_regional = df_holidays_regional.rename(columns={"es_feriado": "es_feriado_regional",
                                                            "es_evento": "es_evento_regional"})

df_holidays_regional = (
    df_holidays_regional.groupby(["date", "locale_name"])
    .agg({"es_feriado_regional": "max", "es_evento_regional": "max"})
    .reset_index()
)

print("Regional")
print(df_holidays_regional["es_feriado_regional"].unique())
print(df_holidays_regional["es_evento_regional"].unique())
df_holidays_regional = df_holidays_regional.drop(columns= "es_evento_regional")

df_holidays_regional.to_csv(ruta_base / "Datasets/Modificados/holidays_regional.csv", index=False)

# --- LOCAL ---
df_holidays_local = df_holidays_events[df_holidays_events["locale"] == "Local"]
df_holidays_local = df_holidays_local.drop(columns= ["locale"])
df_holidays_local = df_holidays_local.rename(columns= {"es_feriado": "es_feriado_local",
                                                       "es_evento": "es_evento_local"})

df_holidays_local = (
    df_holidays_local.groupby(["date", "locale_name"])
    .agg({"es_feriado_local": "max", "es_evento_local": "max"})
    .reset_index()
)

print("Local")
print(df_holidays_local["es_feriado_local"].unique())
print(df_holidays_local["es_evento_local"].unique())
df_holidays_local = df_holidays_local.drop(columns= "es_evento_local")

df_holidays_local.to_csv(ruta_base/"Datasets/Modificados/holidays_local.csv", index= False)

Nacional
[ True False]
[False  True]
Regional
[ True]
[False]
Local
[ True False]
[False]


Las columnas de Evento solo registran Eventos a nivel nacional, por ende, es_evento quedará colo una sola columna dentro del dataset concatenado